In [3]:
# Import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

## Read CSV

In [4]:
customer = pd.read_csv("../data/customer.csv")
df_success = pd.read_csv("../data/df_success.csv")

## Clean data

In [5]:
df_success["InvoiceDate"] = pd.to_datetime(df_success["InvoiceDate"])
now  = df_success["InvoiceDate"].max() + pd.Timedelta(days = 1)
now

Timestamp('2010-12-10 20:01:00')

In [6]:
mean_country = customer.groupby("Country").agg({
    "Monetary" : "mean"
})
number_country = customer.groupby("Country").size()
country = customer.groupby("Country").agg({
    "Country" : "first"
})
country["Average"] = mean_country["Monetary"]
country["Count"] = number_country
country = country.drop("Country", axis = 1)
country = country.reset_index()
country.to_csv("../data/country.csv")
country.head()

,Country,Average,Count
0,Australia,2203.019286,14
1,Austria,1341.433000,10
2,Bahrain,402.985000,2
3,Belgium,1681.608125,16
4,Brazil,268.270000,1


In [7]:
country.describe()

,Average,Count
count,37.000000,37.000000
mean,4294.100760,116.594595
std,11541.475558,651.229327
min,140.390000,1.000000
25%,1323.320000,1.000000
50%,1783.900000,5.000000
75%,3224.234615,13.000000
max,71361.812000,3970.000000


In [8]:
df_success_tmp = df_success.merge(country, on = "Country", how = "left")
df_success_tmp.index = df_success.index
df_success_tmp.head()

,Unnamed: 0,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Amount,Average,Count
0,0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,1867.514386,3970
1,1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,1867.514386,3970
2,2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,1867.514386,3970
3,3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,1867.514386,3970
4,4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,1867.514386,3970


In [9]:
df_success_tmp["InvoiceDate"].min()

Timestamp('2009-12-01 07:45:00')

In [10]:
df_success_tmp["InvoiceDate"].max()

Timestamp('2010-12-09 20:01:00')

In [11]:
first_6_months = df_success[df_success["InvoiceDate"] < (now - pd.Timedelta(days = 180))]
rfm_first_6_months = first_6_months.groupby("Customer ID").agg({
    "InvoiceDate" : lambda x : (now - x.max()).days,
    "Amount" : "sum",
    "Invoice" : "nunique",
    "Country" : 'first'
})
rfm_first_6_months.columns = ["Recency", "Monetary","Frequency","Country"] 
rfm_first_6_months = rfm_first_6_months.reset_index()
rfm_first_6_months = rfm_first_6_months.merge(country, on = "Country", how = "left")
rfm_first_6_months = rfm_first_6_months.drop("Country", axis = 1)
rfm_first_6_months.to_csv("../data/rfm_first_6_months.csv")
rfm_first_6_months.head()

,Customer ID,Recency,Monetary,Frequency,Average,Count
0,12346.0,283,230.55,10,1867.514386,3970
1,12349.0,206,1268.52,2,1369.743636,11
2,12355.0,203,488.21,1,402.985000,2
3,12358.0,186,1697.93,2,1341.433000,10
4,12359.0,262,1522.23,4,1627.107143,7


In [12]:
last_6_months = df_success[df_success["InvoiceDate"] >= (now - pd.Timedelta(days = 180))]
rfm_last_6_months = last_6_months.groupby("Customer ID")["Amount"].sum()
rfm_last_6_months = pd.DataFrame(rfm_last_6_months, columns = ["Amount"])
rfm_last_6_months.head()

,Amount
Customer ID,
12346.0,142.31
12347.0,1323.32
12348.0,222.16
12349.0,1402.62
12351.0,300.93


In [13]:
full_time = rfm_first_6_months.merge(rfm_last_6_months, on = "Customer ID", how = "left")
full_time["Amount"] = full_time["Amount"].fillna(0)
full_time= full_time.drop("Customer ID", axis = 1)
full_time

,Recency,Monetary,Frequency,Average,Count,Amount
0,283,230.55,10,1867.514386,3970,142.31
1,206,1268.52,2,1369.743636,11,1402.62
2,203,488.21,1,402.985000,2,0.00
3,186,1697.93,2,1341.433000,10,1021.08
4,262,1522.23,4,1627.107143,7,1041.13
...,...,...,...,...,...,...
2825,213,120.32,1,1867.514386,3970,0.00
2826,257,354.42,3,1867.514386,3970,287.35
2827,296,427.00,1,1867.514386,3970,0.00
2828,359,462.95,1,1867.514386,3970,833.48


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X = full_time.drop("Amount", axis = 1)
y = full_time["Amount"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2,random_state = 42)
randomfr = RandomForestRegressor(n_estimators=200,max_depth=10, random_state=42)
randomfr.fit(X_train,y_train)
randomfr.score(X_test,y_test)

0.5139720945220692

In [16]:
import joblib

joblib.dump({
    "du_doan_tien" : randomfr
},"../models/regressor.pkl")

['../models/regressor.pkl']